# Atividade: tratamento de uma base de clientes

Complete as células com código pandas. A base tratada deve ficar adequada para análise e aprendizado de máquina.

In [52]:
# Importe a biblioteca pandas.
import pandas as pd
import numpy as np

In [53]:
# Carregue o arquivo clientes_sem_tratamento.csv em um DataFrame.
df = pd.read_csv('clientes_sem_tratamento.csv')

In [54]:
# Inspecione as primeiras e as últimas linhas, as dimensões, os nomes das colunas e os tipos de dados.
print("Primeiras Linhas")
print(df.head())
print("\nUltimas linhas")
print(df.tail())
print("\nDimensões")
print(df.shape)
print("\nNome das colunas")
print(df.columns.tolist())
print("\nTipo de dados")
print(df.dtypes)

Primeiras Linhas
   id_cliente           nome                  email   idade          cidade  \
0           1    Ana Paula     ANA.PAULA@email.com       22       sao paulo   
1           2     Bruno Lima   bruno.lima@email.com      35        Campinas   
2           3    Carla Souza                    NaN      29          Santos   
3           4    Diego Alves  diego.alves@email.com  trinta        Sorocaba   
4           5    Elisa Rocha  ELISA.ROCHA@EMAIL.COM      41  Ribeirão Preto   

      estado renda_mensal data_cadastro ativo  
0         sp  R$ 2.500,50    01/02/2026   Sim  
1         SP      4200.00    2026-02-05     S  
2  São Paulo     3.100,75    07-02-2026   sim  
3         SP  R$ 5.000,00    10/02/2026   Sim  
4         sp          NaN    2026/02/12     1  

Ultimas linhas
    id_cliente             nome                       email idade  \
17          18     Patrícia Luz      patricia.luz@email.com    52   
18          19  Rafael Teixeira   rafael.teixeira@email.com    -4 

In [55]:
# Conte os valores ausentes de cada coluna e procure strings que também representem ausência, como 'não informado'.
print("\nAusentes tradicionais:\n", df.isna().sum())
for col in df.select_dtypes(include=['object', 'string']).columns:
    print(f"\nValores únicos na coluna {col}:", df[col].unique()[:10])


Ausentes tradicionais:
 id_cliente       0
nome             0
email            1
idade            1
cidade           1
estado           0
renda_mensal     1
data_cadastro    0
ativo            0
dtype: int64

Valores únicos na coluna nome: <StringArray>
[ '  Ana Paula  ',     'Bruno Lima',    'Carla Souza',    'Diego Alves',
    'Elisa Rocha',   'Fábio Mendes', 'Gabriela Nunes',   'Hugo Martins',
      'Ana Paula',   'Igor Freitas']
Length: 10, dtype: str

Valores únicos na coluna email: <StringArray>
[    'ANA.PAULA@email.com ',     'bruno.lima@email.com',
                        nan,    'diego.alves@email.com',
    'ELISA.ROCHA@EMAIL.COM',   'fabio.mendes@email.com',
 'gabriela.nunes@email.com',       'hugo.martins@email',
      'ana.paula@email.com',   'igor.freitas@email.com']
Length: 10, dtype: str

Valores únicos na coluna idade: <StringArray>
['22', '35', '29', 'trinta', '41', '27', '33', '38', '150', '31']
Length: 10, dtype: str

Valores únicos na coluna cidade: <StringArray>


In [56]:
# Identifique registros totalmente duplicados e IDs repetidos. Remova a cópia repetida do cliente de id 2.
print("\nRegistros totalmente duplicados:", df.duplicated().sum())
print("IDs repetidos:", df['id_cliente'].duplicated().sum())

# Removendo a cópia do ID 2
# Se houver duplicata exata ou de ID, podemos usar drop_duplicates:
df = df.drop_duplicates(subset=['id_cliente'], keep='first')


Registros totalmente duplicados: 1
IDs repetidos: 1


In [57]:
# Remova espaços extras no início e no fim das colunas textuais.
colunas_texto = df.select_dtypes(include=['object', 'string', 'str']).columns
for col in colunas_texto:
    df[col] = df[col].astype(str).str.strip()

In [58]:
# Padronize os nomes de pessoas com iniciais maiúsculas e os e-mails com letras minúsculas.
if 'nome' in df.columns:
    df['nome'] = df['nome'].str.title()
if 'email' in df.columns:
    df['email'] = df['email'].str.lower()
print("Nomes e emails padronizados")
df[['nome','email']]

Nomes e emails padronizados


,nome,email
0,Ana Paula,ana.paula@email.com
1,Bruno Lima,bruno.lima@email.com
2,Carla Souza,NaN
3,Diego Alves,diego.alves@email.com
4,Elisa Rocha,elisa.rocha@email.com
5,Fábio Mendes,fabio.mendes@email.com
6,Gabriela Nunes,gabriela.nunes@email.com
7,Hugo Martins,hugo.martins@email
9,Ana Paula,ana.paula@email.com
10,Igor Freitas,igor.freitas@email.com


In [59]:
# Localize e remova e-mails inválidos e cadastros duplicados pelo mesmo e-mail. Preserve o valor ausente do e-mail do cliente 3.
import re

# Padrão simples de validação de email
email_pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
# Marcar emails inválidos (exceto NaN)
df['email_valido'] = df['email'].apply(lambda x: bool(re.match(email_pattern, str(x))) if pd.notna(x) else True)
print(f"Emails inválidos encontrados: {(~df['email_valido']).sum()}")
# Remover emails inválidos (mas não remover cliente 3 se email for NaN)
df = df[df['email_valido'] | (df['id_cliente'] == 3)]
df = df.drop('email_valido', axis=1)

# Remover duplicatas de email (preservando NaN do cliente 3)
df_sem_na = df[df['email'].notna()]
df_com_na = df[df['email'].isna()]
df_sem_na = df_sem_na.drop_duplicates(subset='email', keep='first')
df = pd.concat([df_sem_na, df_com_na]).sort_index()
print(f"Dimensões após limpeza de emails: {df.shape}")


Emails inválidos encontrados: 1
Dimensões após limpeza de emails: (19, 9)


In [60]:
# Converta idade para número. Remova idades impossíveis e preencha a idade ausente com a mediana das idades válidas.
# Definindo para numerico
df['idade'] = pd.to_numeric(df['idade'], errors='coerce')

# Remover idades impossiveis (fora do range 0-120)
df.loc[(df['idade'] < 0) | (df['idade'] > 120), 'idade'] = np.nan
print(f"Total de idades ausentes ou impossíveis: {df['idade'].isna().sum()}")

# Preencha os valores ausentes com a mediana e converta para tipo inteiro (int)
mediana_idade = df['idade'].median()
df['idade'] = df['idade'].fillna(mediana_idade).astype(int)
print(f"Mediana das idades válidas: {mediana_idade}")

# Verificação rápida
print(f"Idades ausentes após o tratamento: {df['idade'].isna().sum()}")


Total de idades ausentes ou impossíveis: 4
Mediana das idades válidas: 34.0
Idades ausentes após o tratamento: 0


In [ ]:
# Limpe renda_mensal: remova 'R$', pontos de milhar e espaços; troque a vírgula decimal por ponto; converta para número.


In [ ]:
# Considere rendas negativas e 'não informado' como ausentes. Preencha rendas ausentes com a mediana das rendas válidas.

In [ ]:
# Converta data_cadastro para data, aceitando os formatos existentes. Remova registros cuja data seja impossível.

In [ ]:
# Padronize cidade: corrija caixa, acentuação e grafias equivalentes. Preencha cidade ausente com 'Não informado'.

In [ ]:
# Padronize estado com duas letras maiúsculas. Investigue e remova o registro incompatível com as cidades paulistas da base.

In [ ]:
# Converta as diferentes representações de ativo para os valores booleanos True e False.

In [ ]:
# Reordene pelo id_cliente, redefina o índice e confira novamente dimensões, tipos, ausências, duplicatas e estatísticas.

In [ ]:
# Compare seu resultado com clientes_tratados.csv e explique qualquer diferença encontrada.

In [ ]:
# Salve o DataFrame final em um novo arquivo CSV, sem gravar o índice. Não sobrescreva as bases fornecidas.